<a href="https://colab.research.google.com/github/XavierElon/HackerRankSolutions/blob/master/Copy_of_hw2_text_classification_for_NLP_OMSCS_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Text Classification

In this assignment, we explore three approaches to text classification:
- **Naive Bayes** — A probabilistic classifier based on Bayes' theorem
- **Logistic Regression** — A neural network-based binary classifier
- **Multinomial Logistic Regression** — An extension for multi-class classification

We examine both binary classification (sentiment analysis) and multi-class classification (topic categorization) using two datasets:
- [IMDb Movie Review Sentiment](http://ai.stanford.edu/~amaas/data/sentiment/) — 50K reviews, binary labels (positive/negative)
- [AG News Topics](https://huggingface.co/datasets/ag_news) — News articles across 4 categories

**Tips for Success:**
- Read all provided code—training and evaluation loops illustrate key PyTorch patterns.
- If your model learns (loss decreases) but accuracy plateaus, try adding `nn.Dropout` layers before the final linear layer to improve generalization.

In [1]:
# start time - notebook execution
import time
start_nb = time.time()

# Environment Setup

Configure the runtime environment and import required packages.

In [2]:
import nltk
import numpy as np
import os
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.optim as optim

# Initialize the Autograder

In [4]:
# import the autograder tests
import hw2_tests as ag

# Functions for cleaning up raw texts and tokenizing the corpus

We perform text preprocessing that includes: removing HTML tags, making text lower case, stemming, and disposing of stopwords.
In the end, we will split the entire dataset into training, validation and test sets.

In [ ]:
# Stemming the text
def simple_stemmer(text):
    ps=nltk.porter.PorterStemmer()
    text= [ps.stem(word) for word in text]
    return text

In [ ]:
stopwords_english = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"]
print(stopwords_english)

#removing the stopwords
def remove_stopwords(text, stopword_list):
    tokens = [token.strip() for token in text]
    filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    return filtered_tokens

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'no

In [ ]:
def tokenize_and_clean(line, stem_and_remove_stop_words = True):

    line = re.sub(r"<.*?>", "", line).strip() # remove all HTML tags
    line = re.sub(r'[^a-zA-Z0-9]', ' ', line) # remove punc
    line = line.lower().split()  # lower case
    if stem_and_remove_stop_words:
        line = remove_stopwords(line, stopwords_english)
        line = simple_stemmer(line)

    return line

# Download and unpack the sentiment data



We use the IMDb Dataset for binary sentiment classification, providing 25K highly polar reviews for training and 25K for testing (balanced positive/negative classes).

**Dataset folder structure:**
```
aclImdb/
├── train/
│   ├── pos/
│   └── neg/
└── test/
    ├── pos/
    └── neg/
```

In [ ]:
# check if dataset is downloaded
if not os.path.isfile('aclImdb_v1.tar'):
    print("Downloading dataset...")
    !wget http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !gunzip aclImdb_v1.tar.gz
    !tar -xvf aclImdb_v1.tar

Load in the text from the folders.

In [ ]:
def load_text_from_folders(path, file_list, dataset, samples = 2000, stem_and_remove_stop_words = True):
    """Read set of files from given directory and save returned lines to list.

    Parameters
    ----------
    path : str
        Absolute or relative path to given file (or set of files).
    file_list: list
        List of files names to read.
    dataset: list
        List that stores read lines.
    samples: int
        Number of samples in the output
    """
    for i, file in enumerate(file_list):
        if i >= samples:
            break
        with open(os.path.join(path, file), 'r', encoding='utf8') as text:
            contents = text.read()
            contents_tokenized = tokenize_and_clean(contents, stem_and_remove_stop_words=stem_and_remove_stop_words)
            dataset.append(contents_tokenized)

# Creating Training and Test Sets

This step creates four arrays:

- `train_pos` — Training instances with positive sentiment (label=1)
- `train_neg` — Training instances with negative sentiment (label=0)
- `test_pos` — Test instances with positive sentiment
- `test_neg` — Test instances with negative sentiment

In [ ]:
# Path to dataset location
path = 'aclImdb/'

# Create lists that will contain read lines
train_pos, train_neg, test_pos, test_neg = [], [], [], []

# Create a dictionary of paths and lists that store lines (key: value = path: list)
sets_dict = {'train/pos/': train_pos, 'train/neg/': train_neg,
             'test/pos/': test_pos, 'test/neg/': test_neg}

# Load the data
for dataset in sets_dict:
  file_list = [f for f in sorted(os.listdir(os.path.join(path, dataset))) if f.endswith('.txt')]
  load_text_from_folders(os.path.join(path, dataset), file_list, sets_dict[dataset])

Convert the text data into Pandas DataFrames for easier manipulation. A `DataFrame` provides a tabular structure with labeled columns. We create separate DataFrames for training, testing, and a combined view.

In [ ]:
# Concatenate training and testing examples into one dataset
TRAIN = pd.concat([pd.DataFrame({'review': train_pos, 'label':1}),
                     pd.DataFrame({'review': train_neg, 'label':0})],
                     axis=0, ignore_index=True)

TEST = pd.concat([pd.DataFrame({'review': test_pos, 'label':1}),
                    pd.DataFrame({'review': test_neg, 'label':0})],
                    axis=0, ignore_index=True)

ALL = pd.concat([TRAIN, TEST])

Look at the data.

This is a summary of the data. We see that the data is balanced between labels.

In [ ]:
TRAIN.label.value_counts()

,count
label,
1,2000
0,2000


This is the first few rows of the training set:

In [ ]:
TRAIN.head()

,review,label
0,"[bromwel, high, cartoon, comedi, ran, time, pr...",1
1,"[homeless, houseless, georg, carlin, state, is...",1
2,"[brilliant, act, lesley, ann, warren, best, dr...",1
3,"[easili, underr, film, inn, brook, cannon, sur...",1
4,"[typic, mel, brook, film, much, less, slapstic...",1


# Building the Vocabulary

We construct a vocabulary—a lookup table mapping each unique word to an integer index. This is necessary because neural networks operate on numerical inputs, not strings.

Each word index enables construction of one-hot or bag-of-words vectors of dimension $|V|$, where $|V|$ is the vocabulary size.

In [ ]:
class Vocab:
    def __init__(self, name):
        self.name = name
        self._word2index = {}
        self._word2count = {}
        self._index2word = {}
        self._n_words = 0

    def get_words(self):
      return list(self._word2count.keys())

    def num_words(self):
      return self._n_words

    def word2index(self, word):
      return self._word2index[word]

    def index2word(self, word):
      return self._index2word[word]

    def word2count(self, word):
      return self._word2count[word]

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self._word2index:
            self._word2index[word] = self._n_words
            self._word2count[word] = 1
            self._index2word[self._n_words] = word
            self._n_words += 1
        else:
            self._word2count[word] += 1

Make a vocab object.

In [ ]:
VOCAB = Vocab("imdb")
VOCAB_SIZE = 1000
NUM_LABELS = 2

Load the first ```n``` frequent words in the vocabulary. Do this by sorting by frequency and then truncating.

In [ ]:
# Get word frequency counts
word_freq_dict = {}   # key = word, value = frequency
for review in ALL['review']:
  for word in review:
    if word in word_freq_dict:
      word_freq_dict[word] += 1
    else:
      word_freq_dict[word] = 1

# Get a list of (word, freq) tuples sorted by frequency
kv_list = []  # list of word-freq tuples so can sort
for (k,v) in word_freq_dict.items():
  kv_list.append((k,v))
sorted_kv_list = sorted(kv_list, key=lambda x: x[1], reverse=True)

# Load top n words in to vocab object
for word, freq in sorted_kv_list[:VOCAB_SIZE]:
  VOCAB.add_word(word)

# Naive Bayes

Naive Bayes is a probabilistic classifier based on Bayes' theorem, computing the probability of a class given observed features.

**Bayes' Theorem:**

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Or equivalently:

$$\text{Posterior} = \frac{\text{Likelihood} \times \text{Prior}}{\text{Evidence}}$$

Using word presence as features, create an array of features for each review. Each review will thus be an array of size ```len(vocab)``` where each index in the array is a token number and the value in that position is whether the token is present in the review. There will be ```num_rows``` arrays, making a ```num_rows x len(vocab)``` 2D array.

This function creates a bag of words. It returns a vector where each element is a count of the words in the sentence corresponding to the word index.

In [ ]:
def make_bow(sentence):
    vec = torch.zeros(VOCAB_SIZE, dtype=torch.float64)
    for word in sentence:
        if word not in VOCAB.get_words():
            continue
        vec[VOCAB.word2index(word)] += 1
    return vec.view(1, -1)

Prepare the feature matrices:

- `X_TRAIN` — Shape: `(num_train_samples, vocab_size)`. Each row is a binary presence vector where position $i$ indicates whether vocabulary word $i$ appears in the review.
- `X_TEST` — Same structure for test data.

Each row represents features $\phi_1, \phi_2, \ldots, \phi_{|V|}$ treated as conditionally independent given the class label.

In [ ]:
# Vectorize text reviews to numbers
# Make empty vectors
X_TRAIN = np.zeros((len(TRAIN), VOCAB_SIZE))
X_TEST = np.zeros((len(TEST), VOCAB_SIZE))

# Load in frequency counts
for i, row in TRAIN.iterrows():
    X_TRAIN[i] = np.array(make_bow(row['review'])) > 0 # The > 0 converts to presence instead of counts

for i, row in TEST.iterrows():
    X_TEST[i] = np.array(make_bow(row['review'])) > 0 # The > 0 converts to presence instead of counts

# The labels
Y_TRAIN = np.array(TRAIN['label'])
Y_TEST = np.array(TEST['label'])

What you want to do is to compute probabilities over the training data and then apply those probabilities to the testing examples. Use the Bayes formula to compute $P_{\rm test}(L_{+}|\phi_{0:|V|})$ and $P_{\rm test}(L_{-}|\phi_{0:|V|})$ for each review. Classify examples based on whether one probability is higher than another. That is, $sign(P_{\rm test}(L_{+}|\phi_{0:|V|}) - P_{\rm test}(L_{-}|\phi_{0:|V|}))$ indicates a positive review when greater than 0 and a negative review when less than 0.

**Implementation Hint:** Avoid explicit Python loops. Use NumPy's broadcasting and aggregation functions (`.sum(axis=...)`, `.mean(axis=...)`, boolean indexing) to compute probabilities across all features simultaneously. For example, `X_TRAIN[Y_TRAIN == 1]` selects all positive-class training samples.

Step 1: Compute the positive label condition:
$P(L_{+}|\phi_{0:|V|}) = P(\phi_{0:|V|}|L_{+})P(L_{+}) / P(\phi_{0:|V|})$

In [ ]:
def prob_pos_given_features(x_train, y_train):
  log_probs = np.array([0] * x_train.shape[1])
  ### BEGIN SOLUTION

  # Training data for positive samples
  is_positive = y_train == 1
  n_positive = is_positive.sum()
  X_positive = x_train[is_positive]

  # Calculate feature counts and apply Laplace smoothing
  feature_counts = X_positive.sum(axis=0)
  smoothed_probs = (feature_counts + 1.0) / (n_positive + 2.0)
  log_probs = np.log(smoothed_probs)

  ### END SOLUTION
  return log_probs

Step 2: Compute the negative label condition:
$P(L_{-}|\phi_{0:|V|}) = P(\phi_{0:|V|}|L_{-})P(L_{-}) / P(\phi_{0:|V|})$

In [ ]:
def prob_neg_given_features(x_train, y_train):
  log_probs = np.array([0] * x_train.shape[1])
  ### BEGIN SOLUTION

  # Filter negative samples
  is_negative = y_train == 0
  n_negative = is_negative.sum()
  x_negative = x_train[is_negative]

  # Calculate feature counts and apply Laplace smoothing
  feature_counts = x_negative.sum(axis=0)
  smoothed_probs = (feature_counts + 1.0) / (n_negative + 2.0)
  log_probs = np.log(smoothed_probs)

  ### END SOLUTION
  return log_probs

In [ ]:
pos_probs = prob_pos_given_features(X_TRAIN, Y_TRAIN)
neg_probs = prob_neg_given_features(X_TRAIN, Y_TRAIN)

Step 3: Make a label prediction. Subtract (in log scale) the positive from the negative. If the result is greater than zero then it is a prediction of `+` label. If the result is less than zero then we make a prediction of `-` label.

In [ ]:
def naive_bayes(x, pos_probs, neg_probs):
  label = 0
  ### BEGIN SOLUTION

  # Prevent exp() from return 1.0
  safe_pos = np.clip(pos_probs, -500, 0)
  safe_neg = np.clip(neg_probs, -500, 0)

  # Calculate log probabilities for feature not present in sample
  log_p_absent_pos = np.log(1 - np.exp(safe_pos))
  log_p_absent_neg = np.log(1 - np.exp(safe_neg))

  # Calculate total log-likelihood for each class
  score_pos = np.sum(x * pos_probs + (1 - x) * log_p_absent_pos)
  score_neg = np.sum(x * neg_probs + (1 - x) * log_p_absent_neg)

  # Compare and make final decision
  label = 1 if score_pos > score_neg else 0

  ### END SOLUTION
  return label

# Naive Bayes Test (20 Points)

In [ ]:
# student check - accuracies >= 78% will receive full credit (no credit for less than 78%)
ag.test_naive_bayes(X_TRAIN, Y_TRAIN, X_TEST, Y_TEST)

Accuracy:  0.84225
Test A: 20/20


# Logistic Regression - Part 1

We will be using a neural network to perform logistic regression. We will use word counts as the input feature vector.


Reload the data, but use word counts instead of word presence.

In [ ]:
# Randomize the data
TRAIN = TRAIN.sample(frac=1).reset_index(drop=True)
TEST = TEST.sample(frac=1).reset_index(drop=True)

# Vectorize text reviews to numbers
X_TRAIN = np.zeros((len(TRAIN), VOCAB_SIZE))
X_TEST = np.zeros((len(TEST), VOCAB_SIZE))

for i, row in TRAIN.iterrows():
  X_TRAIN[i] = np.array(make_bow(row['review']))

for i, row in TEST.iterrows():
  X_TEST[i] = np.array(make_bow(row['review']))

Y_TRAIN = np.array(TRAIN['label'])
Y_TEST = np.array(TEST['label'])

Implement a logistic regression classifier as a PyTorch neural network.

**Architecture Requirements:**
- Input: Bag-of-words vector of size `vocab_size`
- Output: Single neuron with sigmoid activation (values in [0, 1])
- Parameters: Exactly `vocab_size` weights + 1 bias (for binary classification)

For binary classification, one output neuron suffices—values near 0 indicate negative sentiment, values near 1 indicate positive sentiment.

In [ ]:
# Defining neural network structure
class BoWClassifier(nn.Module):  # inheriting from nn.Module!

  def __init__(self, num_labels, vocab_size):
    super(BoWClassifier, self).__init__()
    # BEGIN SOLUTION

    # Init single linear layer to map BoW vector to a logit score. 1 is used for binary classification
    self.linear = nn.Linear(vocab_size, 1)

    # END SOLUTION

  def forward(self, bow_vec):
    # BEGIN SOLUTION

    # Apply sigmoid to linear layer to make output in range of 0 to 1
    out = torch.sigmoid(self.linear(bow_vec))

    # END SOLUTION
    return out

In [ ]:
# Initialize the model
# Use one label because the head can signify a 1 or 0 because of the sigmoid.
bow_nn_model = BoWClassifier(NUM_LABELS-1, VOCAB_SIZE)

This function should return two tensors. The first, containing training data, shoud be of size ```batch_size x vocab_size``` for the ```i```th batch. The second should be a list of labels of size ```batch_size```. Both tensors should be of type ```dtype=torch.float```.

In [ ]:
def get_batch(i, batch_size, x_data, y_data):
  # Make some empty tensors
  x = None
  y = None
  ### BEGIN SOLUTION

  # Calculate sliding window for current batch
  start = i * batch_size
  end = start + batch_size

  # Slice data and convert to tensors with float precision to ennsure tensors match expect dtype for linear layer
  x = torch.tensor(x_data[start:end], dtype=torch.float)
  y = torch.tensor(y_data[start:end], dtype=torch.float)

  ### END SOLUTION
  return x, y

# Logistic Regression - Part 1 Test (20 Points)

In [ ]:
# student check
ag.test_batch_output_shape(get_batch, X_TRAIN, Y_TRAIN, VOCAB_SIZE)

Output shape looks good!
Test B: 5/5


In [ ]:
# student check - your model must have the expected number of layers to receive full credit, no credit otherwise
ag.check_bow_architecture(bow_nn_model)

Model has the expected number of layers.
Test C: 5/5
First layer is a Linear layer.
Test D: 5/5


In [ ]:
# student check
ag.test_forward_pass_shape(X_TRAIN, Y_TRAIN, bow_nn_model)

Forward pass output shape looks good!
Test E: 5/5


# Logistic Regression - Part 2

Create a dataset as an array of (X_train, label).

Complete ```get_batch(i)``` and set ```batch_size``` and ```num_epochs```.

Training loop will call ```get_batch()``` with the iteration number and do everything else.


In [ ]:
# Train the model
def train(model, train_data, test_data, epochs, batch_size):
  n_iter = len(train_data) // batch_size
  print(n_iter, 'batches per epoch')
  # Loss Function
  loss_function = nn.BCELoss()
  # Optimizer initlialization
  optimizer = optim.SGD(bow_nn_model.parameters(), lr=0.1)

  for epoch in range(epochs):
    # Make BOW vector for input features and target label
    for i in range(n_iter):
      x, y = get_batch(i, batch_size, train_data, test_data)

      # Step 3. Run the forward pass.
      y_hat = model(x)
      y_hat = y_hat.reshape(-1)

      # Step 4. Compute the loss, gradients, and update the parameters by
      loss = loss_function(y_hat,y)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      if (epoch+1)%10 == 0 and (i+1) == n_iter:
        print('epoch:', epoch+1,',loss =',loss.item(), ', training accuracy =',(torch.round(y_hat)==y).float().mean())
  return model

In [ ]:
# It's ok to modify this cell.
BATCH_SIZE = 100
N_EPOCHS = 1500

In [ ]:
try:
    bow_nn_model = train(bow_nn_model, X_TRAIN, Y_TRAIN, N_EPOCHS, BATCH_SIZE)
except:
    print("Training failed. Please check your code.")

40 batches per epoch
epoch: 10 ,loss = 0.39349648356437683 , training accuracy = tensor(0.8600)
epoch: 20 ,loss = 0.33835190534591675 , training accuracy = tensor(0.8900)
epoch: 30 ,loss = 0.3044029474258423 , training accuracy = tensor(0.8900)
epoch: 40 ,loss = 0.2816143333911896 , training accuracy = tensor(0.8900)
epoch: 50 ,loss = 0.2656153738498688 , training accuracy = tensor(0.8900)
epoch: 60 ,loss = 0.2535930871963501 , training accuracy = tensor(0.8900)
epoch: 70 ,loss = 0.24400368332862854 , training accuracy = tensor(0.9100)
epoch: 80 ,loss = 0.23602579534053802 , training accuracy = tensor(0.9100)
epoch: 90 ,loss = 0.22919349372386932 , training accuracy = tensor(0.9200)
epoch: 100 ,loss = 0.22322151064872742 , training accuracy = tensor(0.9200)
epoch: 110 ,loss = 0.2179236263036728 , training accuracy = tensor(0.9200)
epoch: 120 ,loss = 0.21317140758037567 , training accuracy = tensor(0.9200)
epoch: 130 ,loss = 0.20887213945388794 , training accuracy = tensor(0.9200)
epoch

# Logistic Regression - Part 2 Test (20 Points)

In [ ]:
# student check - accuracies >= 78% will receive full credit (no credit for less than 78%)
ag.test_model_accuracy_lr(TEST, bow_nn_model)

              precision    recall  f1-score   support

           0       0.82      0.82      0.82      2000
           1       0.82      0.81      0.82      2000

    accuracy                           0.82      4000
   macro avg       0.82      0.82      0.82      4000
weighted avg       0.82      0.82      0.82      4000

Accuracy:  0.8
Test F: 20/20


# Multinomial Regression

Load data.

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

Unlike the bag-of-words approach, we now use pre-trained [GLoVe](https://nlp.stanford.edu/projects/glove/) word embeddings. GLoVe maps each word to a dense 100-dimensional vector where semantically similar words have similar vectors.

**Why GLoVe?**
- The AG News corpus has ~400K unique words—one-hot encoding would be prohibitively sparse
- Pre-trained embeddings capture semantic relationships learned from massive corpora
- Input shape changes from `(batch_size, vocab_size)` to `(batch_size, max_words, 100)`

In [ ]:
!pip install gensim
import gensim.downloader

In [ ]:
glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')
VOCAB_SIZE = len(glove_vectors.vectors)
EMBEDDING_DIM = 100

This function will embed the dataset into sequences of 100-dimension vectors.

In [ ]:
# pad dataset to a maximum review length in words
MAX_LEN = 50

def get_glove_seq(review, max_len):
  seq = np.zeros((max_len, 100))
  for i, word in enumerate(review):
    if i < max_len and word in glove_vectors:
      seq[i] = glove_vectors[word]
  return seq

In [ ]:
news_data_train = load_dataset("ag_news", split="train").shuffle()
news_data_test = load_dataset("ag_news", split="test").shuffle()
NEWS_TRAIN = pd.DataFrame(news_data_train)[:5000]
NEWS_TEST = pd.DataFrame(news_data_test)[:5000]
NUM_LABELS = 4

In [ ]:
NEWS_TEST.head()

,text,label
0,"Broadcaster Donates #36;325,000 to GOP (AP) A...",0
1,Hard-running Dillon #39;s difference in making...,1
2,News Corporation to Buy 20 Presses from MAN Ro...,2
3,AOL Shuns Microsoft Anti-Spam Technology Add A...,3
4,Training is the Key to Defibrillator Success B...,3


Train/Test Sets using GloVe embeddings.

In [ ]:
# Vectorize text reviews to numbers
X_NEWS_TRAIN = np.zeros((len(NEWS_TRAIN), MAX_LEN, 100))
X_NEWS_TEST = np.zeros((len(NEWS_TEST), MAX_LEN, 100))

for i, row in NEWS_TRAIN.iterrows():
  X_NEWS_TRAIN[i] = get_glove_seq(tokenize_and_clean(row['text'], stem_and_remove_stop_words=False), MAX_LEN)

for i, row in NEWS_TEST.iterrows():
  X_NEWS_TEST[i] = get_glove_seq(tokenize_and_clean(row['text'], stem_and_remove_stop_words=False), MAX_LEN)

Y_NEWS_TRAIN = np.array(NEWS_TRAIN['label'])
Y_NEWS_TEST = np.array(NEWS_TEST['label'])
NUM_LABELS = 4

In [ ]:
# Defining neural network structure
class MultinomialBoWClassifier(nn.Module):  # inheriting from nn.Module!
  def __init__(self, max_word_len, embedding_dim, num_labels):
    super(MultinomialBoWClassifier, self).__init__()
    self.max_word_len = max_word_len
    self.embedding_dim = embedding_dim
    self.num_labels = num_labels
    ### BEGIN SOLUTION

    # Map averaged word embedding to final scores
    self.linear = nn.Linear(embedding_dim, num_labels)

    # Use dropout to prevent over relying on specific embeddings
    self.dropout = nn.Dropout(p=0.3)

    ### END SOLUTION

  def forward(self, x):
    out = None
    ### BEGIN SOLUTION

    pooled = x.mean(dim=1)

    # Apply regularization before final classification
    dropped_out = self.dropout(pooled)

    out = self.linear(dropped_out)

    ### END SOLUTION
    return out

In [ ]:
multibow_model = MultinomialBoWClassifier(max_word_len=MAX_LEN, embedding_dim=EMBEDDING_DIM, num_labels=NUM_LABELS)

In [ ]:
# Train the model
def train(model, x_train_data, y_train_data, epochs, batch_size, lr, weight_decay):
  print('Training Started!')
  optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
  criterion = nn.CrossEntropyLoss()
  n_iter = len(x_train_data) // batch_size
  print(n_iter, 'batches per epoch')

  for epoch in range(epochs):
    num_correct = 0
    total_loss = 0.0
    model.train()

    for i in range(n_iter):
      x, y = get_batch(i, batch_size, x_train_data, y_train_data)
      x = x
      y = y.long()

      y_hat = model(x)
      loss = criterion(y_hat, y)
      total_loss += loss
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      if (epoch+1)%10 == 0 and (i+1) == n_iter:
        print('epoch:', epoch+1,',loss =',loss.item(), ', training accuracy =',(y_hat.argmax(dim=1)==y).float().mean().item())

In [ ]:
# It's ok to modify this cell.
BATCH_SIZE = 10
N_EPOCHS = 100
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-2

In [ ]:
try:
    train(multibow_model, X_NEWS_TRAIN, Y_NEWS_TRAIN, N_EPOCHS, BATCH_SIZE, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
except:
    print("Training failed. Please check your code.")

Training Started!
500 batches per epoch
epoch: 10 ,loss = 0.7433266043663025 , training accuracy = 0.6000000238418579
epoch: 20 ,loss = 0.504718005657196 , training accuracy = 0.800000011920929
epoch: 30 ,loss = 0.6490519642829895 , training accuracy = 1.0
epoch: 40 ,loss = 0.6428335905075073 , training accuracy = 0.8999999761581421
epoch: 50 ,loss = 0.6253818273544312 , training accuracy = 0.8999999761581421
epoch: 60 ,loss = 0.6654692888259888 , training accuracy = 0.800000011920929
epoch: 70 ,loss = 0.5592200756072998 , training accuracy = 0.8999999761581421
epoch: 80 ,loss = 0.5816346406936646 , training accuracy = 0.8999999761581421
epoch: 90 ,loss = 0.575041651725769 , training accuracy = 1.0
epoch: 100 ,loss = 0.570099413394928 , training accuracy = 0.8999999761581421


# Multinomial Regression - Test (40 Points)

In [ ]:
# student check - accuracies >= 80% will receive full credit (no credit for less than 80%)
ag.test_model_accuracy_mr(X_NEWS_TEST, Y_NEWS_TEST, multibow_model)

Accuracy:  0.8610000014305115
Test G: 40/40


# Submission

Submit this `.ipynb` file to Gradescope for grading.

**Before submitting, verify:**
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Notebook execution time < 30 minutes
- [ ] All autograder tests pass with required accuracy thresholds

## Final Grade

In [ ]:
# student check
ag.FINAL_GRADE()

Your projected points for this assignment is 200/100.

NOTE: THIS IS NOT YOUR FINAL GRADE. YOUR FINAL GRADE FOR THIS ASSIGNMENT WILL BE AT LEAST 200 OR MORE, BUT NOT LESS



## Notebook Runtime

In [ ]:
# end time - notebook execution
end_nb = time.time()
# print notebook execution time in minutes
print("Notebook execution time in minutes =", (end_nb - start_nb)/60)
# warn student if notebook execution time is greater than 30 minutes
if (end_nb - start_nb)/60 > 30:
  print("WARNING: Notebook execution time is greater than 30 minutes. Your submission may not complete auto-grading on Gradescope. Please optimize your code to reduce the notebook execution time.")

Notebook execution time in minutes = 4.393875535329183
